### Задание

Создайте систему компьютерного зрения, которая будет определять тип геометрической фигуры. Используя подготовленную базу и шаблон ноутбука проведите серию экспериментов по перебору гиперпараметров нейронной сети, распознающей три категории изображений (треугольник, круг, квадрат).

1. Поменяйте количество нейронов в сети, используя следующие значения:

- один слой 10 нейронов
- один слой 100 нейронов
- один слой 5000 нейронов.

2. Поменяйте активационную функцию в скрытых слоях с `relu` на `linear`.
3. Поменяйте размеры batch_size:
- 10
- 100
- 1000

4. Выведите на экран получившиеся точности.

Всего должно получиться 18 комбинаций указанных параметров.

Создайте сравнительную таблицу по результатам проведенных тестов.

## Подключение библиотек

In [68]:
# Подключение класса для создания нейронной сети прямого распространения
from tensorflow.keras.models import Sequential
# Подключение класса для создания полносвязного слоя
from tensorflow.keras.layers import Dense
# Подключение оптимизатора
from tensorflow.keras.optimizers import Adam
# Подключение утилит для to_categorical
from tensorflow.keras import utils
# Подключение библиотеки для загрузки изображений
from tensorflow.keras.preprocessing import image
# Подключение библиотеки для работы с массивами
import numpy as np
# Подключение библиотек для отрисовки изображений
import matplotlib.pyplot as plt
# Подключение модуля для работы с файлами
import os
# Вывод изображения в ноутбуке, а не в консоли или файле
import random
%matplotlib inline

## Загрузка датасета

In [69]:
# Загрузка датасета из облака
import gdown
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l3/hw_light.zip', None, quiet=True)

'hw_light.zip'

In [70]:
# Распаковываем архив hw_light.zip в папку hw_light
!unzip -q hw_light.zip

replace hw_light/0/1.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace hw_light/0/10.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace hw_light/0/10.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


## Подготовка данных

In [71]:
# Путь к директории с базой
base_dir = '/content/hw_light'
# Создание пустого списка для загрузки изображений обучающей выборки
x_train = []
# Создание списка для меток классов
y_train = []
# Задание высоты и ширины загружаемых изображений
img_height = 20
img_width = 20
# Перебор папок в директории базы
for patch in os.listdir(base_dir):
    # Перебор файлов в папках
    for img in os.listdir(base_dir + '/' + patch):
        # Добавление в список изображений текущей картинки
        x_train.append(image.img_to_array(image.load_img(base_dir + '/' + patch + '/' + img,
                                                    target_size=(img_height, img_width),
                                                    color_mode='grayscale')))
        # Добавление в массив меток, соответствующих классам
        if patch == '0':
            y_train.append(0)
        elif patch == '3':
            y_train.append(1)
        else:
            y_train.append(2)

# Преобразование в numpy-массив загруженных изображений и меток классов
x_train = np.array(x_train)
y_train = np.array(y_train)
# Вывод размерностей
print('Размер массива x_train', x_train.shape)
print('Размер массива y_train', y_train.shape)

Размер массива x_train (302, 20, 20, 1)
Размер массива y_train (302,)


## Подготовка данных перед созданием модели

In [72]:
CLASS_COUNT = 3
x_train = x_train.reshape(x_train.shape[0], -1)
x_train = x_train.astype('float32') / 255.

In [73]:
y_train = utils.to_categorical(y_train, CLASS_COUNT)

In [74]:
print(f"x_train: {x_train.shape}")
print(f"y_train: {y_train.shape}")

x_train: (302, 400)
y_train: (302, 3)


## Функция для создания и обучения моделей

In [75]:

def create_model(unit, activ):
    model = Sequential()

    model.add(Dense(400, input_dim=400, activation=activ))
    model.add(Dense(unit, activation=activ))
    model.add(Dense(CLASS_COUNT, activation='softmax'))
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    model.summary()

    return model

## Использование Relu

## 10 нейронов и 10 batch_size

In [76]:
model1 = create_model(10, 'relu')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_23"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_69 (Dense)                │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_70 (Dense)                │ (None, 10)             │         4,010 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_71 (Dense)                │ (None, 3)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 164,443 (642.36 KB)

 Trainable params: 164,443 (642.36 KB)

 Non-trainable params: 0 (0.00 B)

In [77]:
history1 = model1.fit(x_train,
            y_train,
            batch_size=10,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 55ms/step - accuracy: 0.5851 - loss: 0.9791 - val_accuracy: 0.4098 - val_loss: 1.2749
Epoch 2/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7884 - loss: 0.6039 - val_accuracy: 0.6393 - val_loss: 1.0743
Epoch 3/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7718 - loss: 0.5684 - val_accuracy: 0.7049 - val_loss: 0.8986
Epoch 4/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8755 - loss: 0.3914 - val_accuracy: 0.7377 - val_loss: 0.7600
Epoch 5/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8506 - loss: 0.3971 - val_accuracy: 0.6557 - val_loss: 1.0548
Epoch 6/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8672 - loss: 0.3311 - val_accuracy: 0.6885 - val_loss: 0.8889
Epoch 7/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8880 - loss: 0.2878 - val_accuracy: 0.6557 - val_loss: 1.1017
Epoch 8/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9336 - loss: 0.2085 - val_accuracy: 0.6230 - val_loss

In [78]:
val_acc1 = history1.history['val_accuracy'][-1]

## 100 нейронов и 10 batch_size

In [79]:
model2 = create_model(100, 'relu')

Model: "sequential_24"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_72 (Dense)                │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_73 (Dense)                │ (None, 100)            │        40,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_74 (Dense)                │ (None, 3)              │           303 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 200,803 (784.39 KB)

 Trainable params: 200,803 (784.39 KB)

 Non-trainable params: 0 (0.00 B)

In [80]:
history2 = model2.fit(x_train,
            y_train,
            batch_size=10,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 57ms/step - accuracy: 0.6141 - loss: 0.8032 - val_accuracy: 0.7049 - val_loss: 0.8316
Epoch 2/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7261 - loss: 0.6514 - val_accuracy: 0.6393 - val_loss: 0.8160
Epoch 3/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8091 - loss: 0.4821 - val_accuracy: 0.6557 - val_loss: 0.8067
Epoch 4/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8506 - loss: 0.3918 - val_accuracy: 0.6230 - val_loss: 1.2849
Epoch 5/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8838 - loss: 0.3399 - val_accuracy: 0.5902 - val_loss: 1.6547
Epoch 6/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8506 - loss: 0.3827 - val_accuracy: 0.5574 - val_loss: 1.3924
Epoch 7/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8672 - loss: 0.3181 - val_accuracy: 0.7377 - val_loss: 0.6705
Epoch 8/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9585 - loss: 0.1804 - val_accuracy: 0.6230 - val_loss

In [81]:
val_acc2 = history2.history['val_accuracy'][-1]

## 5000 нейронов и 10 batch_size

In [82]:
model3 = create_model(5000, 'relu')

Model: "sequential_25"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_75 (Dense)                │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_76 (Dense)                │ (None, 5000)           │     2,005,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_77 (Dense)                │ (None, 3)              │        15,003 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,180,403 (8.32 MB)

 Trainable params: 2,180,403 (8.32 MB)

 Non-trainable params: 0 (0.00 B)

In [83]:
history3 = model3.fit(x_train,
            y_train,
            batch_size=10,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 89ms/step - accuracy: 0.6017 - loss: 1.0596 - val_accuracy: 0.5574 - val_loss: 1.4577
Epoch 2/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7801 - loss: 0.5959 - val_accuracy: 0.6393 - val_loss: 0.9721
Epoch 3/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8548 - loss: 0.3358 - val_accuracy: 0.4590 - val_loss: 1.6469
Epoch 4/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8755 - loss: 0.2921 - val_accuracy: 0.7049 - val_loss: 0.9113
Epoch 5/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9544 - loss: 0.1607 - val_accuracy: 0.6230 - val_loss: 1.5025
Epoch 6/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9336 - loss: 0.1802 - val_accuracy: 0.6557 - val_loss: 1.3706
Epoch 7/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9834 - loss: 0.0615 - val_accuracy: 0.6393 - val_loss: 1.9112
Epoch 8/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9461 - loss: 0.1070 - val_accuracy: 0.7049 - val_los

In [84]:
val_acc3 = history3.history['val_accuracy'][-1]

## 10 нейронов и 100 batch_size

In [85]:
model4 = create_model(10, 'relu')

Model: "sequential_26"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_78 (Dense)                │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_79 (Dense)                │ (None, 10)             │         4,010 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_80 (Dense)                │ (None, 3)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 164,443 (642.36 KB)

 Trainable params: 164,443 (642.36 KB)

 Non-trainable params: 0 (0.00 B)

In [86]:
history4 = model4.fit(x_train,
            y_train,
            batch_size=100,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 507ms/step - accuracy: 0.3402 - loss: 1.1193 - val_accuracy: 0.0000e+00 - val_loss: 1.4806
Epoch 2/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5519 - loss: 0.8896 - val_accuracy: 0.0000e+00 - val_loss: 1.3835
Epoch 3/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.6432 - loss: 0.7778 - val_accuracy: 0.0000e+00 - val_loss: 1.3995
Epoch 4/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.6680 - loss: 0.7087 - val_accuracy: 0.0328 - val_loss: 1.4899
Epoch 5/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.7137 - loss: 0.6529 - val_accuracy: 0.1475 - val_loss: 1.4499
Epoch 6/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.7510 - loss: 0.6128 - val_accuracy: 0.3770 - val_loss: 1.2194
Epoch 7/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8050 - loss: 0.5663 - val_accuracy: 0.4426 - val_loss: 1.3141
Epoch 8/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8257 - loss: 0.5343 - val_accuracy: 0.3770 - val_

In [87]:
val_acc4 = history4.history['val_accuracy'][-1]

## 100 нейронов и 100 batch_size

In [88]:
model5 = create_model(100, 'relu')

Model: "sequential_27"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_81 (Dense)                │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_82 (Dense)                │ (None, 100)            │        40,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_83 (Dense)                │ (None, 3)              │           303 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 200,803 (784.39 KB)

 Trainable params: 200,803 (784.39 KB)

 Non-trainable params: 0 (0.00 B)

In [89]:
history5 = model5.fit(x_train,
            y_train,
            batch_size=100,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 509ms/step - accuracy: 0.4606 - loss: 1.1993 - val_accuracy: 0.0000e+00 - val_loss: 1.9723
Epoch 2/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.6805 - loss: 0.7451 - val_accuracy: 0.5902 - val_loss: 0.8877
Epoch 3/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.7759 - loss: 0.6297 - val_accuracy: 0.5082 - val_loss: 1.5439
Epoch 4/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.7718 - loss: 0.5695 - val_accuracy: 0.5410 - val_loss: 1.1715
Epoch 5/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8631 - loss: 0.4678 - val_accuracy: 0.6393 - val_loss: 1.0734
Epoch 6/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8382 - loss: 0.4343 - val_accuracy: 0.5738 - val_loss: 1.1950
Epoch 7/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8755 - loss: 0.4013 - val_accuracy: 0.6557 - val_loss: 1.0754
Epoch 8/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8631 - loss: 0.3539 - val_accuracy: 0.5902 - val_loss: 1.

In [90]:
val_acc5 = history5.history['val_accuracy'][-1]

## 5000 нейронов и 100 batch_size

In [91]:
model6 = create_model(5000, 'relu')

Model: "sequential_28"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_84 (Dense)                │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_85 (Dense)                │ (None, 5000)           │     2,005,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_86 (Dense)                │ (None, 3)              │        15,003 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,180,403 (8.32 MB)

 Trainable params: 2,180,403 (8.32 MB)

 Non-trainable params: 0 (0.00 B)

In [92]:
history6 = model6.fit(x_train,
            y_train,
            batch_size=100,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 724ms/step - accuracy: 0.4689 - loss: 1.3849 - val_accuracy: 0.6885 - val_loss: 0.8155
Epoch 2/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.5477 - loss: 1.0243 - val_accuracy: 0.4918 - val_loss: 1.0305
Epoch 3/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.6598 - loss: 0.7071 - val_accuracy: 0.5738 - val_loss: 1.2058
Epoch 4/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.7178 - loss: 0.6654 - val_accuracy: 0.5902 - val_loss: 0.9616
Epoch 5/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8589 - loss: 0.4996 - val_accuracy: 0.6066 - val_loss: 0.8325
Epoch 6/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9046 - loss: 0.3976 - val_accuracy: 0.6393 - val_loss: 0.9564
Epoch 7/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8631 - loss: 0.3346 - val_accuracy: 0.5574 - val_loss: 1.3655
Epoch 8/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9170 - loss: 0.2678 - val_accuracy: 0.7541 - val_loss: 0.7011

In [93]:
val_acc6 = history6.history['val_accuracy'][-1]

## 10 нейронов и 1000 batch_size

In [94]:
model7 = create_model(10, 'relu')

Model: "sequential_29"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_87 (Dense)                │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_88 (Dense)                │ (None, 10)             │         4,010 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_89 (Dense)                │ (None, 3)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 164,443 (642.36 KB)

 Trainable params: 164,443 (642.36 KB)

 Non-trainable params: 0 (0.00 B)

In [95]:
history7 = model7.fit(x_train,
            y_train,
            batch_size=1000,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.4232 - loss: 1.0812 - val_accuracy: 0.0000e+00 - val_loss: 2.5396
Epoch 2/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.4232 - loss: 1.0142 - val_accuracy: 0.0164 - val_loss: 1.8186
Epoch 3/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.4315 - loss: 0.8974 - val_accuracy: 0.1311 - val_loss: 1.3842
Epoch 4/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.4398 - loss: 1.0301 - val_accuracy: 0.6066 - val_loss: 1.0294
Epoch 5/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.6722 - loss: 0.8920 - val_accuracy: 0.5082 - val_loss: 1.2215
Epoch 6/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.5934 - loss: 0.8743 - val_accuracy: 0.3607 - val_loss: 1.5364
Epoch 7/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.6722 - loss: 0.8163 - val_accuracy: 0.0984 - val_loss: 1.8248
Epoch 8/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.6556 - loss: 0.7500 - val_accuracy: 0.0164 - val_loss: 1.98

In [96]:
val_acc7 = history7.history['val_accuracy'][-1]

### 100 нейронов и 1000 batch_size

In [97]:
model8 = create_model(100, 'relu')

Model: "sequential_30"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_90 (Dense)                │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_91 (Dense)                │ (None, 100)            │        40,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_92 (Dense)                │ (None, 3)              │           303 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 200,803 (784.39 KB)

 Trainable params: 200,803 (784.39 KB)

 Non-trainable params: 0 (0.00 B)

In [98]:
history8 = model8.fit(x_train,
            y_train,
            batch_size=1000,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.2033 - loss: 1.9859 - val_accuracy: 0.0000e+00 - val_loss: 1.7560
Epoch 2/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.5021 - loss: 0.9604 - val_accuracy: 0.0000e+00 - val_loss: 2.4941
Epoch 3/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.4315 - loss: 1.1199 - val_accuracy: 0.0000e+00 - val_loss: 2.4242
Epoch 4/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.5768 - loss: 0.9238 - val_accuracy: 0.0000e+00 - val_loss: 2.1836
Epoch 5/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.5394 - loss: 0.8943 - val_accuracy: 0.0656 - val_loss: 1.7060
Epoch 6/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.5851 - loss: 0.8027 - val_accuracy: 0.1967 - val_loss: 1.2504
Epoch 7/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.7469 - loss: 0.7199 - val_accuracy: 0.3770 - val_loss: 1.0107
Epoch 8/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.7759 - loss: 0.7348 - val_accuracy: 0.4590 - val

In [99]:
val_acc8 = history8.history['val_accuracy'][-1]

## 5000 нейронов и 1000 batch_size

In [100]:
model9 = create_model(5000, 'relu')

Model: "sequential_31"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_93 (Dense)                │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_94 (Dense)                │ (None, 5000)           │     2,005,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_95 (Dense)                │ (None, 3)              │        15,003 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,180,403 (8.32 MB)

 Trainable params: 2,180,403 (8.32 MB)

 Non-trainable params: 0 (0.00 B)

In [101]:
history9 = model9.fit(x_train,
            y_train,
            batch_size=1000,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.3154 - loss: 1.1394 - val_accuracy: 0.0000e+00 - val_loss: 6.5312
Epoch 2/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - accuracy: 0.4647 - loss: 1.5811 - val_accuracy: 0.0000e+00 - val_loss: 6.2065
Epoch 3/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 309ms/step - accuracy: 0.4232 - loss: 2.0984 - val_accuracy: 0.0492 - val_loss: 2.7036
Epoch 4/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 279ms/step - accuracy: 0.7095 - loss: 0.8419 - val_accuracy: 0.5574 - val_loss: 1.1263
Epoch 5/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - accuracy: 0.4689 - loss: 0.9644 - val_accuracy: 0.7377 - val_loss: 0.5702
Epoch 6/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - accuracy: 0.5270 - loss: 0.9559 - val_accuracy: 0.8033 - val_loss: 0.5047
Epoch 7/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - accuracy: 0.6266 - loss: 0.7699 - val_accuracy: 0.7049 - val_loss: 0.6492
Epoch 8/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.8382 - loss: 0.6245 - val_accuracy: 0.5738 - val_

In [102]:
val_acc9 = history9.history['val_accuracy'][-1]

## Использование linear

## 10 нейронов и 10 batch_size

In [103]:
model10 = create_model(10, 'linear')

Model: "sequential_32"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_96 (Dense)                │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_97 (Dense)                │ (None, 10)             │         4,010 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_98 (Dense)                │ (None, 3)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 164,443 (642.36 KB)

 Trainable params: 164,443 (642.36 KB)

 Non-trainable params: 0 (0.00 B)

In [104]:
history10 = model10.fit(x_train,
            y_train,
            batch_size=10,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.4440 - loss: 1.9701 - val_accuracy: 0.0656 - val_loss: 3.8644
Epoch 2/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6846 - loss: 0.8869 - val_accuracy: 0.4262 - val_loss: 2.6112
Epoch 3/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8091 - loss: 0.6340 - val_accuracy: 0.5738 - val_loss: 1.7492
Epoch 4/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8091 - loss: 0.5284 - val_accuracy: 0.6557 - val_loss: 1.2620
Epoch 5/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8465 - loss: 0.4831 - val_accuracy: 0.3443 - val_loss: 3.1352
Epoch 6/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8008 - loss: 0.5104 - val_accuracy: 0.6393 - val_loss: 1.3618
Epoch 7/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8589 - loss: 0.3996 - val_accuracy: 0.7377 - val_loss: 0.7489
Epoch 8/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7593 - loss: 0.6233 - val_accuracy: 0.5738 - val_loss

In [105]:
val_acc10 = history10.history['val_accuracy'][-1]

## 100 нейронов и 10 batch_size

In [106]:
model11 = create_model(100, 'linear')

Model: "sequential_33"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_99 (Dense)                │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_100 (Dense)               │ (None, 100)            │        40,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_101 (Dense)               │ (None, 3)              │           303 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 200,803 (784.39 KB)

 Trainable params: 200,803 (784.39 KB)

 Non-trainable params: 0 (0.00 B)

In [107]:
history11 = model11.fit(x_train,
            y_train,
            batch_size=10,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 67ms/step - accuracy: 0.5436 - loss: 1.5280 - val_accuracy: 0.5738 - val_loss: 1.7630
Epoch 2/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7012 - loss: 0.7743 - val_accuracy: 0.4426 - val_loss: 2.0434
Epoch 3/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7593 - loss: 0.6386 - val_accuracy: 0.6066 - val_loss: 1.6193
Epoch 4/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8133 - loss: 0.5170 - val_accuracy: 0.5082 - val_loss: 1.7462
Epoch 5/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7801 - loss: 0.5779 - val_accuracy: 0.1803 - val_loss: 3.1529
Epoch 6/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7842 - loss: 0.5169 - val_accuracy: 0.6066 - val_loss: 1.5635
Epoch 7/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8050 - loss: 0.4659 - val_accuracy: 0.7213 - val_loss: 0.9216
Epoch 8/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8465 - loss: 0.4363 - val_accuracy: 0.6393 - val_loss

In [108]:
val_acc11 = history11.history['val_accuracy'][-1]

## 5000 нейронов и 10 batch_size

In [109]:
model12 = create_model(5000, 'linear')

Model: "sequential_34"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_102 (Dense)               │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_103 (Dense)               │ (None, 5000)           │     2,005,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_104 (Dense)               │ (None, 3)              │        15,003 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,180,403 (8.32 MB)

 Trainable params: 2,180,403 (8.32 MB)

 Non-trainable params: 0 (0.00 B)

In [110]:
history12 = model12.fit(x_train,
            y_train,
            batch_size=10,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 69ms/step - accuracy: 0.4813 - loss: 3.4065 - val_accuracy: 0.5410 - val_loss: 2.5233
Epoch 2/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6888 - loss: 0.8533 - val_accuracy: 0.6393 - val_loss: 2.1510
Epoch 3/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7967 - loss: 0.6255 - val_accuracy: 0.3279 - val_loss: 2.6098
Epoch 4/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7759 - loss: 0.6363 - val_accuracy: 0.4918 - val_loss: 2.5271
Epoch 5/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7967 - loss: 0.5790 - val_accuracy: 0.5246 - val_loss: 2.6364
Epoch 6/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7842 - loss: 0.4881 - val_accuracy: 0.5738 - val_loss: 2.3356
Epoch 7/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7967 - loss: 0.5048 - val_accuracy: 0.7869 - val_loss: 0.7132
Epoch 8/13
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6888 - loss: 0.9285 - val_accuracy: 0.5574 - val_loss

In [111]:
val_acc12 = history12.history['val_accuracy'][-1]

## 10 нейронов и 100 batch_size

In [112]:
model13 = create_model(10, 'linear')

Model: "sequential_35"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_105 (Dense)               │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_106 (Dense)               │ (None, 10)             │         4,010 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_107 (Dense)               │ (None, 3)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 164,443 (642.36 KB)

 Trainable params: 164,443 (642.36 KB)

 Non-trainable params: 0 (0.00 B)

In [113]:
history13 = model13.fit(x_train,
            y_train,
            batch_size=100,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 479ms/step - accuracy: 0.4606 - loss: 3.6591 - val_accuracy: 0.9836 - val_loss: 0.0940
Epoch 2/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.2116 - loss: 3.5767 - val_accuracy: 0.0000e+00 - val_loss: 5.3377
Epoch 3/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.4896 - loss: 1.8569 - val_accuracy: 0.0000e+00 - val_loss: 10.6526
Epoch 4/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5602 - loss: 2.0591 - val_accuracy: 0.0000e+00 - val_loss: 7.8813
Epoch 5/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5021 - loss: 1.7569 - val_accuracy: 0.0000e+00 - val_loss: 5.4707
Epoch 6/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.6556 - loss: 1.0310 - val_accuracy: 0.4754 - val_loss: 1.4424
Epoch 7/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.6349 - loss: 0.8903 - val_accuracy: 0.6885 - val_loss: 0.8177
Epoch 8/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6763 - loss: 0.8237 - val_accuracy: 0.3443 -

In [114]:
val_acc13 = history13.history['val_accuracy'][-1]

## 100 нейронов и 100 batch_size

In [115]:
model14 = create_model(100, 'linear')

Model: "sequential_36"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_108 (Dense)               │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_109 (Dense)               │ (None, 100)            │        40,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_110 (Dense)               │ (None, 3)              │           303 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 200,803 (784.39 KB)

 Trainable params: 200,803 (784.39 KB)

 Non-trainable params: 0 (0.00 B)

In [116]:
history14 = model14.fit(x_train,
            y_train,
            batch_size=100,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 486ms/step - accuracy: 0.4274 - loss: 2.8492 - val_accuracy: 0.0000e+00 - val_loss: 10.5015
Epoch 2/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.4606 - loss: 2.5313 - val_accuracy: 0.8033 - val_loss: 0.4162
Epoch 3/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.4979 - loss: 2.3251 - val_accuracy: 0.4754 - val_loss: 1.8560
Epoch 4/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.6473 - loss: 1.0399 - val_accuracy: 0.0000e+00 - val_loss: 5.6462
Epoch 5/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6722 - loss: 1.2486 - val_accuracy: 0.1967 - val_loss: 4.1365
Epoch 6/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.6846 - loss: 0.8888 - val_accuracy: 0.6066 - val_loss: 1.4176
Epoch 7/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.7012 - loss: 0.7554 - val_accuracy: 0.6557 - val_loss: 0.9507
Epoch 8/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.7344 - loss: 0.6807 - val_accuracy: 0.5246 - val_los

In [117]:
val_acc14 = history14.history['val_accuracy'][-1]

## 5000 нейронов и 100 batch_size

In [118]:
model15 = create_model(5000, 'linear')

Model: "sequential_37"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_111 (Dense)               │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_112 (Dense)               │ (None, 5000)           │     2,005,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_113 (Dense)               │ (None, 3)              │        15,003 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,180,403 (8.32 MB)

 Trainable params: 2,180,403 (8.32 MB)

 Non-trainable params: 0 (0.00 B)

In [119]:
history15 = model15.fit(x_train,
            y_train,
            batch_size=100,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 480ms/step - accuracy: 0.3237 - loss: 4.8305 - val_accuracy: 0.0000e+00 - val_loss: 9.8705
Epoch 2/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.4481 - loss: 2.6573 - val_accuracy: 0.4262 - val_loss: 2.1351
Epoch 3/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5643 - loss: 1.6372 - val_accuracy: 0.2951 - val_loss: 5.5329
Epoch 4/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5726 - loss: 2.0914 - val_accuracy: 0.4754 - val_loss: 3.3181
Epoch 5/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.6805 - loss: 1.0356 - val_accuracy: 0.6557 - val_loss: 1.3552
Epoch 6/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.6266 - loss: 1.0996 - val_accuracy: 0.6230 - val_loss: 1.7234
Epoch 7/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8174 - loss: 0.5610 - val_accuracy: 0.2623 - val_loss: 3.6091
Epoch 8/13
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.7427 - loss: 0.7221 - val_accuracy: 0.3443 - val_loss: 2.

In [120]:
val_acc15 = history15.history['val_accuracy'][-1]

## 10 нейронов и 1000 batch_size

In [121]:
model16 = create_model(10, 'linear')

Model: "sequential_38"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_114 (Dense)               │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_115 (Dense)               │ (None, 10)             │         4,010 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_116 (Dense)               │ (None, 3)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 164,443 (642.36 KB)

 Trainable params: 164,443 (642.36 KB)

 Non-trainable params: 0 (0.00 B)

In [122]:
history16 = model16.fit(x_train,
            y_train,
            batch_size=1000,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.4149 - loss: 1.1775 - val_accuracy: 0.0000e+00 - val_loss: 3.7976
Epoch 2/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.4232 - loss: 4.4376 - val_accuracy: 0.4262 - val_loss: 0.7686
Epoch 3/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.3693 - loss: 2.7222 - val_accuracy: 0.9672 - val_loss: 0.4982
Epoch 4/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.2614 - loss: 1.2660 - val_accuracy: 0.0000e+00 - val_loss: 2.8695
Epoch 5/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - accuracy: 0.4149 - loss: 2.1909 - val_accuracy: 0.0000e+00 - val_loss: 4.0013
Epoch 6/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.4149 - loss: 1.9754 - val_accuracy: 0.0000e+00 - val_loss: 4.6184
Epoch 7/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - accuracy: 0.6598 - loss: 1.1588 - val_accuracy: 0.0000e+00 - val_loss: 5.5701
Epoch 8/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 0.4398 - loss: 1.6969 - val_accuracy: 0.0000

In [123]:
val_acc16 = history16.history['val_accuracy'][-1]

## 100 нейронов и 1000 batch_size

In [124]:
model17 = create_model(100, 'linear')

Model: "sequential_39"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_117 (Dense)               │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_118 (Dense)               │ (None, 100)            │        40,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_119 (Dense)               │ (None, 3)              │           303 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 200,803 (784.39 KB)

 Trainable params: 200,803 (784.39 KB)

 Non-trainable params: 0 (0.00 B)

In [125]:
history17 = model17.fit(x_train,
            y_train,
            batch_size=1000,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.4813 - loss: 1.1135 - val_accuracy: 0.0000e+00 - val_loss: 10.8791
Epoch 2/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.4149 - loss: 6.3152 - val_accuracy: 0.0000e+00 - val_loss: 6.0949
Epoch 3/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.4440 - loss: 1.6938 - val_accuracy: 0.0000e+00 - val_loss: 5.9323
Epoch 4/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.4232 - loss: 3.6451 - val_accuracy: 0.0164 - val_loss: 2.4416
Epoch 5/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.4232 - loss: 3.5840 - val_accuracy: 1.0000 - val_loss: 0.1220
Epoch 6/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.1701 - loss: 3.8922 - val_accuracy: 0.8525 - val_loss: 0.2726
Epoch 7/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.2324 - loss: 2.4344 - val_accuracy: 0.4262 - val_loss: 1.3387
Epoch 8/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.5519 - loss: 0.9872 - val_accuracy: 0.0656 - val_lo

In [126]:
val_acc17 = history17.history['val_accuracy'][-1]

## 5000 нейронов и 1000 batch_size

In [127]:
model18 = create_model(5000, 'linear')

Model: "sequential_40"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_120 (Dense)               │ (None, 400)            │       160,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_121 (Dense)               │ (None, 5000)           │     2,005,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_122 (Dense)               │ (None, 3)              │        15,003 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,180,403 (8.32 MB)

 Trainable params: 2,180,403 (8.32 MB)

 Non-trainable params: 0 (0.00 B)

In [128]:
history18 = model18.fit(x_train,
            y_train,
            batch_size=1000,
            validation_split=0.2,
            epochs=13,
            verbose=1)

Epoch 1/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.3983 - loss: 1.0930 - val_accuracy: 0.0000e+00 - val_loss: 23.6278
Epoch 2/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5228 - loss: 4.5263 - val_accuracy: 0.0000e+00 - val_loss: 22.6537
Epoch 3/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.4149 - loss: 11.9595 - val_accuracy: 0.0984 - val_loss: 7.5382
Epoch 4/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - accuracy: 0.4440 - loss: 3.8057 - val_accuracy: 0.6393 - val_loss: 1.0635
Epoch 5/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.4481 - loss: 2.6022 - val_accuracy: 0.7869 - val_loss: 0.5676
Epoch 6/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.3278 - loss: 4.3102 - val_accuracy: 0.5738 - val_loss: 1.5528
Epoch 7/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.4896 - loss: 2.6678 - val_accuracy: 0.4918 - val_loss: 2.0678
Epoch 8/13
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.7759 - loss: 0.7263 - val_accuracy: 0.4754 - val_loss:

In [129]:
val_acc18 = history18.history['val_accuracy'][-1]

## Таблица

In [132]:
import pandas as pd

results = pd.DataFrame({
    'Модель': [
        'relu unit 10 batch 10',
        'relu unit 100 batch 10',
        'relu unit 5000 batch 10',

        'relu unit 10 batch 100',
        'relu unit 100 batch 100',
        'relu unit 5000 batch 100',

        'relu unit 10 batch 1000',
        'relu unit 100 batch 1000',
        'relu unit 5000 batch 1000',

        'linear unit 10 batch 10',
        'linear unit 100 batch 10',
        'linear unit 5000 batch 10',

        'linear unit 10 batch 100',
        'linear unit 100 batch 100',
        'linear unit 5000 batch 100',

        'linear unit 10 batch 1000',
        'linear unit 100 batch 1000',
        'linear unit 5000 batch 1000',


    ],
    'Val accuracy': [
        val_acc1, val_acc2, val_acc3,
        val_acc4, val_acc5, val_acc6,
        val_acc7, val_acc8, val_acc9,

        val_acc10, val_acc11, val_acc12,
        val_acc13, val_acc14, val_acc15,
        val_acc16, val_acc17, val_acc18,
    ],
})

results

,Модель,Val accuracy
0,relu unit 10 batch 10,0.655738
1,relu unit 100 batch 10,0.672131
2,relu unit 5000 batch 10,0.672131
3,relu unit 10 batch 100,0.524590
4,relu unit 100 batch 100,0.606557
5,relu unit 5000 batch 100,0.606557
6,relu unit 10 batch 1000,0.524590
7,relu unit 100 batch 1000,0.377049
8,relu unit 5000 batch 1000,0.524590
9,linear unit 10 batch 10,0.524590
